In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import math
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
'''
import warnings
warnings.filterwarnings('ignore')
'''
! pip install Catboost
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
cars_path = os.path.join(path, 'Q1_data.csv')                                  # Builds the full file path: downloaded folder + vehicles.csv.

df_FoodDelivery = pd.read_csv(cars_path)                                                # Reads CSV into a DataFrame named df_cars.

In [ ]:
df_FoodDelivery.shape

In [ ]:
# Task 2: Write your code here:
df_FoodDelivery.head()

In [ ]:
# Task 3: Write your code here:
df_FoodDelivery.info()

In [ ]:
# Task 4: Write your code here:
df_FoodDelivery.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10,5))

plt.hist(df_FoodDelivery['Delivery_Time'].dropna(), color='pink', edgecolor='black' )

plt.title('Food Delivery Time Distribution')
plt.xlabel('Food Delivery Time')
plt.ylabel('Frequency')

plt.show()

In [ ]:
 df_FoodDelivery.head()

In [ ]:
# Task 1: Write your code here:
#Debug
print(f"Dataset Shape Before Dropping: \n\t - { df_FoodDelivery.shape}\n\n")

# Removes the column "Order_ID" from the dataset
df_FoodDelivery = df_FoodDelivery.drop(['Order_ID'], axis=1)

print(f"Dataset Shape After Dropping: \n\t - { df_FoodDelivery.shape}\n\n")
df_FoodDelivery.head()

In [ ]:
# Task 2: Write your code here:

# First we Check Missing

# Is there any missing data?
has_missing = df_FoodDelivery.isnull().values.any()
print("Is There Any Missing Data?")
print("\t- ", has_missing)

# Total missing values in the whole dataset
total_missing = df_FoodDelivery.isnull().sum().sum()
print("\nTotal Missing Values in Dataset:")
print("\t- ", total_missing)

# Missing values per column
missing_per_col = df_FoodDelivery.isnull().sum()
print("\nMissing Values per Column:")
print(missing_per_col)





# Handle them
print('-'*70)
print("Drop Missing")
print("Original shape:", df_FoodDelivery.shape)

# Drop Rows
df_FoodDelivery = df_FoodDelivery.dropna()

print("New shape:", df_FoodDelivery.shape)

print("\nIs There Any Missing Data After Dropping Rows?")
print("\t- ", df_FoodDelivery.isnull().isnull().sum())



In [ ]:
# Task 3: Write your code here:

# Function: Check & Drop Duplicates
def check_duplicates(df):

  duplicates = df.duplicated().sum()                                            # .duplicated(): returns True/False for each row, True if the row is a duplicate of a previous row  -  .sum(): counts how many True values → total number of duplicates
  print(f"Number of Duplicate Samples: {duplicates}")

  if duplicates > 0:                                                            # Checks if duplicates exist (more than 0) ?  If yes, Remove them
    print("\n- Dropping Duplicates...")
    df.drop_duplicates(inplace=True)                                            # inplace=True: the original df is modified directly (no new copy) ; permanently removes duplicates from df
    print("\n*** Duplicates Dropped. ***")

  else:
    print("\n- No Duplicate Samples Found.")


# Fun Call
print("Before Droping Duplicate shape:", df_FoodDelivery.shape)
check_duplicates(df_FoodDelivery)
print("After Droping Duplicate shape:", df_FoodDelivery.shape)

In [ ]:
# Debug
print(f"Before Encoding:\n")
df_FoodDelivery.head(5)

In [ ]:
# Task 4: Write your code here:

# Do we have categorical columns?
categorical_cols = df_FoodDelivery.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))
print("Number of Categorical Columns:", len(list(categorical_cols)))


# Encode them
# Import
# Empty Dictionary to store encoders for each categorical column
label_encoders = {}

for col in categorical_cols:

  # Instantiate Label Enoder
  le = LabelEncoder()                                                           # Creates a LabelEncoder object to store learned label mappings

  # fit_transform()
  df_FoodDelivery[col] = le.fit_transform(df_FoodDelivery[col])                   # 1) fit() learns unique labels  2) transform() converts them into Unique integers

  label_encoders[col] = le                                                      # Stores the encoder in the dictionary using column name as the key


print(f"After Encoding:\n")
df_FoodDelivery.head(5)


In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)


# Select numeric features
numerical_cols = df_FoodDelivery.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
# column counts
print("Number of Numeric features:", len(df_FoodDelivery.columns))


# Scale
# Instantiate StandardScaler
standard_scaler = StandardScaler()

# fit_trnasform()
df_FoodDelivery[numerical_cols] = standard_scaler.fit_transform(df_FoodDelivery[numerical_cols])


# Debug
print(f"\nAfter Scalling ranges ")
df_FoodDelivery.head()

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

#  What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distriabution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

print("Target Distribution: \n\t")
print(df_FoodDelivery['Delivery_Time'].value_counts())

check_target_distribution(df_FoodDelivery, "Delivery_Time")

print( "it imbalnce since values ranges target Differ")

In [ ]:
# Task 1: Write your code here:

# Separate Features and Target

target_column = "Delivery_Time"


X = df_FoodDelivery.drop(target_column, axis=1)
y = df_FoodDelivery[target_column]




# Debug
print("\nDataset Shape Before Splitting:" , df_FoodDelivery.shape)
print("Features X: \n\t", list(X.columns) )
print("\nNumber of Features X: \n\t", X.shape[1] )
print("\nShape of Features X: \n\t", X.shape )
print("")
print("-"*70)
print("")
print("\nTrget Label y: \n\t", y.name )
print("\nNumber of Trget Values: \n\t", y.shape[0] )
print("\nShape of Trget Label y: \n\t", y.shape )

In [ ]:
# Task 2,3,4,5: Write your code here:
'''
Use the correct split: KFold OR StratifiedKFold
Train a RandomForest model
Evaluate using MAE (Mean Absolute Error) ONLY
Print the averaged score across all folds
'''

# Create K-Fold Cross Validation
n_folds = 5                                                                     # K=5 Folds

skf = StratifiedKFold(
    n_splits= n_folds,                                                          # split data into 5 folds
    shuffle= True,                                                              # shuffle data before splitting
    random_state= 42                                                            # reproducible splits
)



# Train RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=100,                                                           # Num of trees -  More trees  improve performance but take more tim
    max_depth=15,                                                               # Maximum depth for each tree (how deep the tree can grow)  -   Controls complexity:   1) Deeper tree → learns more details → may overfit    2) Smaller depth → more general → less overfit
    random_state=42,                                                            # Fixes randomness so results are reproducible (same model each run)   -  Random Forest uses randomness in: sampling rows (bootstrapping) selecting random subset of features
    n_jobs=-1,                                                                  # Use all CPU cores available to speed up training   -  (-1 means “use all cores”)
    class_weight='balanced'
    )



# Storage for RF results for each fold
mse_scores = []
mae_scores = []


# Stratified K-Fold Training Loop
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):          # Uses X and y, Ensures class balance in each fold, Returns indices, not data
  print(f"\nFold {fold_idx + 1}/{n_folds}")

  # Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]                     #    # Select training and testing data using indices
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  # Train
  model.fit(X_train, y_train)

  # Predict
  y_fold_pred = model.predict(X_test)

  # Metrics for this fold
  mse_fold = mean_squared_error(y_test, y_fold_pred)
  mae_fold = mean_absolute_error(y_test, y_fold_pred)

  mse_scores.append(mse_fold)
  mae_scores.append(mae_fold)

  print(f"  MSE : {mse_fold:.2f}")
  print(f"  MAE : {mae_fold:.2f}")
  print("-"*40)


# Final average across folds
print("-"*70)
print("\nAverage across folds")
print(f"  MSE : {np.mean(mse_scores):.2f}")
print(f"  MAE : {np.mean(mae_scores):.2f}")



In [ ]:
# Task 1: Write your code here:
# Stores the feature names (column names of X)
feature_cols = X.columns                                                            # will be used as labels in the plot
print(feature_cols)

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10,5))

plt.hist(y, color='pink', edgecolor='black' )

plt.title('Predicted delivery time  Distribution')
plt.xlabel('Predicted Dilviray Time')
plt.ylabel('Frequency')

plt.show()

In [ ]:
# Task Bonus: Write your code here:

# Dictionary of Models                                                          , where:vvkey = model name (string)  -  value = actual model object (sklearn / lightgbm / catboost)  -   a clean way to train many models using a loop
models = {

  "Random Forest Regressor": RandomForestRegressor(                             # Rnadom Forset = many trees combined together (ensemble) by Voting/Averaging
      n_estimators=200                                                          # number of trees = 200
      ),

  "CatBoost": CatBoostRegressor(                                                # CatBoost = Gradient Boosting Decision Trees, very strong,  handles categorical features well (when present)
      verbose=0                                                                 # no output logs
      )
}

print("Model Comparision between: \n   - ", list(models.keys()) )
print("\nNumber of Modeles we will Compare: \n   - ", len(models) )




# Storage for results
all_results = {}                                                                # Dictionary to store evaluation results for each model

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}                         # For each model name: create a results structure inside all_results




In [ ]:

# Stratified K-Fold Training Loop
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):          # Uses X and y, Ensures class balance in each fold, Returns indices, not data
  print(f"\nFold {fold_idx + 1}/{n_folds}")

  # Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]                     #    # Select training and testing data using indices
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)


# Final average across folds
print("-"*70)
print("\nAverage across folds")
print(f"  MSE : {np.mean(mse_scores):.2f}")
print(f"  MAE : {np.mean(mae_scores):.2f}")

